In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay,
)
 
from imblearn.over_sampling import SMOTENC
 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', "{:.3f}".format)

In [2]:
df = pd.read_csv("loan.csv")
print("Shape of the dataset:",df.shape)
df.head()

Shape of the dataset: (614, 13)


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.000,NaN,360.000,1.000,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.000,128.000,360.000,1.000,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.000,66.000,360.000,1.000,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.000,120.000,360.000,1.000,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.000,141.000,360.000,1.000,Urban,Y


In [3]:
# lowercase column names
df.columns = [col.lower() for col in df.columns]
 
# drop ID column - it carries no information about approval
df = df.drop(columns=["loan_id"])
 
target_col = "loan_status"
 
# Convert Y / N to 1 / 0
df[target_col] = df[target_col].map({"Y": 1, "N": 0})
df.head()

,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,property_area,loan_status
0,Male,No,0,Graduate,No,5849,0.000,NaN,360.000,1.000,Urban,1
1,Male,Yes,1,Graduate,No,4583,1508.000,128.000,360.000,1.000,Rural,0
2,Male,Yes,0,Graduate,Yes,3000,0.000,66.000,360.000,1.000,Urban,1
3,Male,Yes,0,Not Graduate,No,2583,2358.000,120.000,360.000,1.000,Urban,1
4,Male,No,0,Graduate,No,6000,0.000,141.000,360.000,1.000,Urban,1


In [6]:
print("Target distribution:")
print(df[target_col].value_counts(normalize=True))
print("-" * 40)
print("Missing values per column:")
print(df.isnull().sum())
 

Target distribution:
loan_status
1   0.687
0   0.313
Name: proportion, dtype: float64
----------------------------------------
Missing values per column:
gender               13
married               3
dependents           15
education             0
self_employed        32
applicantincome       0
coapplicantincome     0
loanamount           22
loan_amount_term     14
credit_history       50
property_area         0
loan_status           0
dtype: int64


In [9]:
print(df.groupby("credit_history",dropna=False)[target_col].mean())

credit_history
0.000   0.079
1.000   0.796
NaN     0.740
Name: loan_status, dtype: float64


In [10]:
X = df.drop(columns=[target_col])
y = df[target_col]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # keep the same class ratio in train and test
)
 
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
 
X_train_processed = X_train.copy()
X_test_processed = X_test.copy()
 

Train shape: (491, 11)
Test shape : (123, 11)


In [12]:
num_cols = ["applicantincome", "coapplicantincome", "loanamount", "loan_amount_term", "credit_history"]
cat_cols = ["gender", "married", "dependents", "education", "self_employed", "property_area"]
print(num_cols)
print(cat_cols)

['applicantincome', 'coapplicantincome', 'loanamount', 'loan_amount_term', 'credit_history']
['gender', 'married', 'dependents', 'education', 'self_employed', 'property_area']


In [13]:
# 1) missing indicator for credit_history BEFORE imputing (missingness is informative)
for d in (X_train_processed, X_test_processed):
    d["credit_history_missing"] = d["credit_history"].isnull().astype(int)
 
# 2) numerical -> median, categorical -> most frequent
#    fit on TRAIN only, transform both (avoids data leakage)
num_imputer = SimpleImputer(strategy="median")
X_train_processed[num_cols] = num_imputer.fit_transform(X_train_processed[num_cols])
X_test_processed[num_cols] = num_imputer.transform(X_test_processed[num_cols])
 
cat_imputer = SimpleImputer(strategy="most_frequent")
X_train_processed[cat_cols] = cat_imputer.fit_transform(X_train_processed[cat_cols])
X_test_processed[cat_cols] = cat_imputer.transform(X_test_processed[cat_cols])
 
print("Missing values left (train):", X_train_processed.isnull().sum().sum())
print("Missing values left (test) :", X_test_processed.isnull().sum().sum())

Missing values left (train): 0
Missing values left (test) : 0


In [14]:
def add_features(d):
    d = d.copy()
    d["total_income"] = d["applicantincome"] + d["coapplicantincome"]
    d["loan_to_income"] = d["loanamount"] / d["total_income"]
    return d
 
X_train_processed = add_features(X_train_processed)
X_test_processed = add_features(X_test_processed)
 
skewed_cols = ["applicantincome", "coapplicantincome", "loanamount", "total_income", "loan_to_income"]
print("Skew before log1p:")
print(X_train_processed[skewed_cols].skew())
 
for d in (X_train_processed, X_test_processed):
    d[skewed_cols] = np.log1p(d[skewed_cols])
 
print("-" * 40)
print("Skew after log1p:")
print(X_train_processed[skewed_cols].skew())
 

Skew before log1p:
applicantincome     6.546
coapplicantincome   7.606
loanamount          2.791
total_income        5.779
loan_to_income      0.948
dtype: float64
----------------------------------------
Skew after log1p:
applicantincome      0.769
coapplicantincome   -0.138
loanamount          -0.182
total_income         1.171
loan_to_income       0.856
dtype: float64


In [15]:
binary_maps = {
    "gender":        {"Male": 1, "Female": 0},
    "married":       {"Yes": 1, "No": 0},
    "education":     {"Graduate": 1, "Not Graduate": 0},
    "self_employed": {"Yes": 1, "No": 0},
    "dependents":    {"0": 0, "1": 1, "2": 2, "3+": 3},   # ordinal
}
 
for d in (X_train_processed, X_test_processed):
    for col, mapping in binary_maps.items():
        d[col] = d[col].map(mapping)
 
# any unmapped value would become NaN - check it
assert X_train_processed[list(binary_maps)].isnull().sum().sum() == 0
assert X_test_processed[list(binary_maps)].isnull().sum().sum() == 0
 
X_train_processed.head()
 

,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,property_area,credit_history_missing,total_income,loan_to_income
154,1,0,0,1,0,8.088,0.000,3.932,360.000,1.000,Urban,0,8.088,0.015
239,1,1,1,1,0,8.107,0.000,4.575,360.000,1.000,Semiurban,0,8.107,0.029
448,1,1,2,1,0,8.114,7.445,5.017,360.000,0.000,Rural,0,8.527,0.029
471,1,1,1,0,0,7.884,7.314,4.736,180.000,0.000,Rural,0,8.332,0.027
273,1,1,0,1,0,7.871,7.707,5.017,360.000,1.000,Semiurban,0,8.485,0.031


In [16]:
scale_cols = ["applicantincome", "coapplicantincome", "loanamount",
              "loan_amount_term", "total_income", "loan_to_income"]
 
scaler = StandardScaler()
X_train_processed[scale_cols] = scaler.fit_transform(X_train_processed[scale_cols])
X_test_processed[scale_cols] = scaler.transform(X_test_processed[scale_cols])
 
X_train_processed[scale_cols].describe().loc[["mean", "std"]]

,applicantincome,coapplicantincome,loanamount,loan_amount_term,total_income,loan_to_income
mean,-0.000,-0.000,-0.000,-0.000,0.000,-0.000
std,1.001,1.001,1.001,1.001,1.001,1.001


In [17]:
print("Before SMOTENC:")
print(y_train.value_counts())
 
smote_cat_cols = ["gender", "married", "dependents", "education", "self_employed",
                  "credit_history", "credit_history_missing", "property_area"]
 
smote = SMOTENC(categorical_features=smote_cat_cols, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_processed, y_train)
 
print("-" * 40)
print("After SMOTENC:")
print(y_train_res.value_counts())
 
# categorical columns still contain only valid values
for col in ["gender", "credit_history", "property_area"]:
    print(col, sorted(X_train_res[col].unique()))

Before SMOTENC:
loan_status
1    337
0    154
Name: count, dtype: int64
----------------------------------------
After SMOTENC:
loan_status
1    337
0    337
Name: count, dtype: int64
gender [np.int64(0), np.int64(1)]
credit_history [np.float64(0.0), np.float64(1.0)]
property_area ['Rural', 'Semiurban', 'Urban']


In [18]:
ohe = OneHotEncoder(drop="first", sparse_output=False)   # drop one column -> avoids dummy-variable trap
ohe.fit(X_train_res[["property_area"]])
 
def one_hot(d):
    encoded = pd.DataFrame(
        ohe.transform(d[["property_area"]]),
        columns=ohe.get_feature_names_out(["property_area"]),
        index=d.index,
    )
    return pd.concat([d.drop(columns=["property_area"]), encoded], axis=1)
 
X_train_final = one_hot(X_train_res)
X_test_final = one_hot(X_test_processed)
 
print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)
X_train_final.head()

X_train_final: (674, 15)
X_test_final : (123, 15)


,gender,married,dependents,education,self_employed,applicantincome,coapplicantincome,loanamount,loan_amount_term,credit_history,credit_history_missing,total_income,loan_to_income,property_area_Semiurban,property_area_Urban
0,1,0,0,1,0,-0.431,-1.088,-1.896,0.281,1.000,0,-1.087,-1.021,0.000,1.000
1,1,1,1,1,0,-0.401,-1.088,-0.596,0.281,1.000,0,-1.053,0.669,1.000,0.000
2,1,1,2,1,0,-0.389,0.834,0.299,0.281,0.000,0,-0.278,0.761,0.000,0.000
3,1,1,1,0,0,-0.752,0.800,-0.270,-2.487,0.000,0,-0.638,0.453,0.000,0.000
4,1,1,0,1,0,-0.772,0.902,0.299,0.281,1.000,0,-0.355,0.918,1.000,0.000
